In [17]:
from langchain.agents import AgentExecutor, Tool, create_react_agent
from langchain.prompts import PromptTemplate
from langchain.chat_models import ChatOpenAI

import json

In [22]:
import agents.npc as npc
import agents.enemy as enemy

cura = npc.NPC(name="Captain Cura")
with open("benchmarks/enemies/sheets/thugs.json", "r") as fh:
    content = fh.read()
    thug = enemy.Enemy("thug", json.loads(content), "Big Pete")

def talk_to_target(message):
        """Use this to talk to Captain Cura. Use complete sentences!"""
        response = cura.talk(message)
        return response["output"]
    
def talk_to_participant(message):
    """Use this to talk to Big Pete. Use complete sentences!"""
    response = thug.talk(message)
    return response["output"]

tools = [
    Tool(
        name="TalkToParticipant",
        func=talk_to_participant,
        description=f"Call this to to communicate with the participant whose turn it is. Use complete sentences!"
    ),
    Tool(
        name="TalkToTarget",
        func=talk_to_target,
        description=f"Call this to to communicate with a participant who has been targeted by an action. Use complete sentences!"
    ),
]

In [23]:
dm_react_template = """
**You are an experienced dungeon-master for Dungeons & Dragons 5th edition, overseeing an encounter between a player and enemies.**

You have access to the following tools: {tools}

Your role is to fully resolve a single turn. You will ask the participant what they wish to do
and you will determine whether the action is successful. The user will tell you whose turn it is
and what the current state of the battlefield is.

**Procedure for Resolving Turns:**

**Stage One: Request Participant's Action.**
- Ask the participant for the action they wish to take
**Stage Two: Resolving Action (Complete Actions):**
- Collect from the participant all information relevant to the success of the action, including status effects, skill checks, attack rolls, and damage rolls.
- Collect from the target all information relevant to the success of the action, including armor class, status effects, reactions, and resistances.
**Stage Three: Determine Outcome:**
- Use the information you have collected to determin the result of the action.
**Stage Four: Notify Affected Participants:**
- Notify all the participants involved of your determination of the success and effect of the action
**Stage Five: End of Turn:**

For each stage, you should use the following format:

Question: the stage you are on
Thought: you should always think about what to do
Action: the action you should take, should be one of [{tool_names}]
Action Input: the input to the action - the information you hope to obtain
Observation: the response you get from the action
... (this Thought/Action/Action Input/Observation can repeat N times)
Thought: I have everything I need to move on to the next stage of the procedure

Only after ALL stages are complete should you return your final assessment of the turn's outcome. The final assessment
should describe the state of the battle and return the status of all participants in the following format:
  - Name: [[Name]]
  - Current HP: [[HP]]
  - Status Effects: [[Status Effects]]
After the final assessment has been provided, the turn has been fully resolved.

Begin!
Starting state: {input}
Question: Stage One
Thought: {agent_scratchpad}
"""

In [24]:
# Choose the LLM to use
llm = ChatOpenAI()

# Construct the ReAct agent
agent = create_react_agent(llm, tools, PromptTemplate.from_template(dm_react_template))
agent_executor = AgentExecutor(agent=agent, tools=tools, verbose=True,  handle_parsing_errors=True)

In [25]:
inp = json.dumps({
    "to_act": "Big Pete",
    "participants": [
        {"name": "Captain Cura", "HP": 16, "status_effects": []},
        {"name": "Big Pete", "HP": 32, "status_effects": []},
    ],
    "initiative_order": [
        {"participant": "Big Pete", "value": 22},
        {"participant": "Captain Cura", "value": 18}
    ]
})
agent_executor.invoke({"input": inp})



> Entering new AgentExecutor chain...
I need to ask Big Pete what action he wishes to take.
Action: TalkToParticipant
Action Input: "Big Pete"Hey there, Big Pete! How can I assist you today?I'm waiting for Big Pete to tell me what action he wants to take.Invalid Format: Missing 'Action:' after 'Thought:Question: Stage One
Thought: I need to ask Big Pete what action he wishes to take.
Action: TalkToParticipant
Action Input: "Big Pete"

KeyboardInterrupt: 

In [44]:
script = _

{'input': '{"to_act": "Big Pete", "participants": [{"name": "Captain Cura", "HP": 16, "AC": 14, "status_effects": []}, {"name": "Big Pete", "HP": 32, "AC": 15, "status_effects": []}], "initiative_order": [{"participant": "Big Pete", "value": 22}, {"participant": "Captain Cura", "value": 18}]}',
 'output': 'Agent stopped due to iteration limit or time limit.'}